# STEP 5 — 평가 · 보정 · 안전장치

여기가 이 프로젝트의 **진짜 목적지**입니다.

목표가 "정확도 높은 모델"이 아니라 **"보호자에게 의심된다고 말해도 되는 시스템"** 이니까요.
그러려면 세 가지가 필요합니다:

| 필요한 것 | 왜 | 이 노트북의 단계 |
|---|---|---|
| 정직한 성능 숫자 | 부풀려진 정확도로 판단하면 안 됨 | 1, 2 |
| 정직한 **확률** | "신뢰도 72%" 가 진짜 72% 여야 함 | 3 |
| 병변을 보고 있다는 증거 | 배경 보고 맞히면 실사용에서 무너짐 | 4 |
| 모르면 모른다고 하기 | 애매한 사진에 답을 지어내면 안 됨 | 5 |

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    BASE = "/content" if os.path.isdir("/content") else (
           "/kaggle/working" if os.path.isdir("/kaggle/working") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

In [ ]:
from src import labels, split, data, models, train, evaluate, calibrate, explain, infer
from src.config import MODEL_BY_KEY
import torch, json

BEST_CROP = (env.work_root()/"best_crop.txt").read_text().strip() \
            if (env.work_root()/"best_crop.txt").exists() else "m1.5"
BEST_MODEL = "convnextv2_base"     # ← 04 결과에 맞춰 바꾸세요

df = labels.load(env.work_root()/"manifests"/f"manifest_{BEST_CROP}.parquet")
tr, va = split.get_fold(df, 0)
ho = split.get_holdout(df)
print(f"train {len(tr):,} / val {len(va):,} / holdout {len(ho):,}")

In [ ]:
spec = MODEL_BY_KEY[BEST_MODEL]
cfg = CFG(model_name=spec.timm_name, img_size=spec.img_size, exp_name=f"zoo_{BEST_MODEL}")
model = models.load_checkpoint(
    str(env.work_root()/"checkpoints"/f"zoo_{BEST_MODEL}"/"best.pt"),
    spec, len(CLASSES))
print("로드 완료")

## 1. 검증셋 성능

In [ ]:
_, dl_va, _, _ = data.build_loaders(tr, va, cfg, model=model)
_, logits_va, y_va = train.evaluate_loader(model, dl_va, None, "cuda",
                                           len(CLASSES), tta_hflip=True)
rep_va = evaluate.full_report(logits_va, y_va, CLASSES)
rep_va.plot_confusion()

## 2. 확률 보정 ★

신경망은 **자기 확신이 과합니다.** "95% 확신" 이라고 한 예측 100건 중
실제로는 70건만 맞는 게 흔합니다.

우리는 보호자에게 "신뢰도 72%" 같은 숫자를 보여줄 건데, 그게 거짓말이면 안 되죠.

**온도 스케일링**: logits 를 T 로 나누기만 합니다. 파라미터 딱 1개.
예측 순위는 전혀 안 바뀌니 **정확도는 그대로**, 확률만 정직해집니다.

⚠️ T 는 **검증셋**으로 학습하고 **holdout** 에서 효과를 확인합니다.
holdout 으로 T 를 맞추면 그것도 과적합입니다.

In [ ]:
T = calibrate.fit_temperature(logits_va, y_va)
json.dump({"temperature": T},
          open(env.work_root()/"checkpoints"/f"zoo_{BEST_MODEL}"/"temperature.json", "w"))

## 3. Holdout 최종 평가

⚠️ **여기서부터는 되돌릴 수 없습니다.**
holdout 결과를 보고 하이퍼파라미터를 고치면 더 이상 holdout 이 아닙니다.
모든 결정이 끝난 뒤 딱 한 번만 여세요.

In [ ]:
_, dl_ho, _, _ = data.build_loaders(tr, ho, cfg, model=model)
_, logits_ho, y_ho = train.evaluate_loader(model, dl_ho, None, "cuda",
                                           len(CLASSES), tta_hflip=True)
rep_ho = evaluate.full_report(logits_ho, y_ho, CLASSES)

In [ ]:
cal = calibrate.report(logits_va, y_va, logits_ho, y_ho)

In [ ]:
import numpy as np
from src.evaluate import softmax_np

probs_before = softmax_np(logits_ho)
probs_after = calibrate.apply(logits_ho, T)
calibrate.reliability_diagram(probs_before, probs_after, y_ho.numpy())

## 4. Grad-CAM — 필수 검증 게이트 ★★

**정확도가 아무리 좋아도 여기서 통과 못 하면 그 모델은 실패입니다.**

이 데이터는 병변이 이미지의 5% 미만이고 배경이 제각각입니다.
모델이 병변이 아니라 진료대 무늬, 조명, 털 색을 보고 맞힐 수 있고,
그 단서가 클래스와 상관이 있으면 **검증 점수까지 잘 나옵니다.**

숫자로는 절대 못 잡습니다. 그림을 봐야 합니다.

In [ ]:
explain.grid(model, va, cfg, n=8)

In [ ]:
# 틀린 예측만 골라 보기 — 어디를 보고 틀렸는지가 개선의 힌트
import pandas as pd
va2 = va.copy()
va2["pred"] = [CLASSES[i] for i in evaluate.softmax_np(logits_va).argmax(1)]
explain.grid(model, va2, cfg, n=8, only_correct=False)

In [ ]:
# 수치화: CAM 이 실제 병변 박스와 얼마나 겹치는가
overlap = explain.lesion_overlap_score(model, va, cfg, n=150)

### 🚦 게이트 판정

- `median_lift` ≥ 1.3 → 병변을 보고 있음, 통과
- `median_lift` < 1.3 → **배경 학습 의심**. 정확도와 무관하게 재작업

재작업 방향: 크롭 margin 축소 / 배경 증강 강화 / 세그멘테이션 마스킹

## 5. 임계값과 거절(abstention) 설계

"모르면 모른다고 하기"를 수치로 정합니다.

In [ ]:
cr = calibrate.coverage_risk_curve(probs_after, y_ho.numpy())

In [ ]:
thr = calibrate.suggest_abstain_threshold(probs_after, y_ho.numpy(), max_risk=0.20)

### 1단계(정상/이상) 임계값

무증상 데이터가 있어서 2단계 모델을 만든 경우에만 해당합니다.

보호자용 스크리닝에서는 **미탐(병변인데 괜찮다고 함)이 오탐보다 훨씬 나쁩니다.**
그래서 recall 을 먼저 0.95로 고정하고, 그 조건의 precision 을 확인합니다.

In [ ]:
# 무증상(A0)이 클래스에 포함된 경우에만 실행
if "A0" in CLASSES:
    i0 = CLASSES.index("A0")
    score_abnormal = 1 - probs_after[:, i0]      # '이상일 확률'
    y_abnormal = (y_ho.numpy() != i0).astype(int)
    binrep = evaluate.binary_report(score_abnormal, y_abnormal,
                                    target_recall=cfg.target_recall_stage1)
else:
    print("무증상 클래스가 없어 1단계 평가를 건너뜁니다.")
    print("→ 6종 분류 + 저신뢰 거절 방식으로 운용합니다.")

## 6. 실제 사용 시뮬레이션

사용자가 사진 한 장을 올렸을 때 무엇이 보이는지 확인합니다.

In [ ]:
engine = infer.Engine(model, CFG(**{**cfg.to_dict(), "abstain_threshold": thr}),
                     CLASSES, temperature=T)

sample = ho.sample(3, random_state=0)
for _, r in sample.iterrows():
    print("=" * 62)
    print("정답:", r["label"], CLASS_KO.get(r["label"], ""))
    engine.show(r["crop_path"])
    print()

## 7. 최종 리포트 저장

In [ ]:
summary = {
    "model": BEST_MODEL, "crop": BEST_CROP,
    "val": {"macro_f1": rep_va.metrics["macro_f1"],
            "ci": list(rep_va.ci[1:]),
            "per_class_recall": rep_va.metrics["per_class"]["recall"]},
    "holdout": {"macro_f1": rep_ho.metrics["macro_f1"],
                "ci": list(rep_ho.ci[1:]),
                "balanced_acc": rep_ho.metrics["balanced_accuracy"],
                "per_class_recall": rep_ho.metrics["per_class"]["recall"]},
    "calibration": cal,
    "cam_lesion_overlap": overlap,
    "abstain_threshold": thr,
    "coverage_risk": cr,
}
p = env.work_root()/"reports"/f"final_{BEST_MODEL}.json"
p.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print("저장:", p)
print(json.dumps(summary["holdout"], indent=2, ensure_ascii=False))

---
## ✅ 배포 전 체크리스트

- [ ] holdout macro-F1 이 검증셋과 크게 다르지 않다 (차이 크면 과적합)
- [ ] 모든 클래스의 recall 이 0.5 이상이다
- [ ] 보정 후 ECE < 0.10
- [ ] Grad-CAM 이 병변을 보고 있다 (median_lift ≥ 1.3)
- [ ] 저신뢰 거절 임계값이 정해져 있다
- [ ] 모든 출력에 "진단이 아님" 문구가 붙는다

📖 반드시 읽기: [`docs/cautions/03_의료AI_안전설계_원칙.md`](../docs/cautions/03_의료AI_안전설계_원칙.md)